In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_1.py ──
"""
Shared infrastructure for MLFP04 Exercise 1 — Clustering Zoo.

Contains: customer-feature loading & standardisation, metric helpers,
subsampling utilities, and output-directory management.

Technique-specific code (K-means elbow, linkage methods, DBSCAN epsilon
search, HDBSCAN, spectral Laplacian, AutoMLEngine) does NOT belong
here — it lives in the per-technique files under modules/mlfp04/solutions/ex_1/.
"""

import asyncio
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml import ExperimentTracker
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex1_clustering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared random state so every technique file is reproducible
RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════


def load_customers() -> tuple[pl.DataFrame, list[str]]:
    """Load the e-commerce customer dataset and return (df, numeric_feature_cols).

    The dataset (from MLFP03) is ~6K rows of Singapore e-commerce customers
    with recency, frequency, monetary, basket-size, and channel features.
    Clustering is unsupervised segmentation: no labels, just behaviour.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")
    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in ("customer_id",)
    ]
    return customers.drop_nulls(subset=feature_cols), feature_cols


def standardise(
    df: pl.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, StandardScaler]:
    """Return (X_scaled, scaler). Zero mean, unit variance — mandatory for
    all distance-based clustering."""
    X, _, _ = to_sklearn_input(df, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — spectral / hierarchical are O(n^2) or worse
# ════════════════════════════════════════════════════════════════════════


def subsample(
    X: np.ndarray, n: int, seed: int = RANDOM_STATE
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_sub, idx) where idx are the original row indices chosen."""
    rng = np.random.default_rng(seed)
    n = min(n, X.shape[0])
    idx = rng.choice(X.shape[0], n, replace=False)
    return X[idx], idx


# ════════════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════════════


def score_partition(X: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    """Compute silhouette, Calinski-Harabasz, Davies-Bouldin.

    Points with label == -1 (DBSCAN noise) are excluded. Returns NaN
    fields if fewer than 2 valid clusters remain.
    """
    valid = labels != -1
    labs = labels[valid]
    data = X[valid]
    if data.shape[0] < 2 or len(set(labs.tolist())) < 2:
        return {
            "n_clusters": len(set(labs.tolist())),
            "silhouette": float("nan"),
            "calinski_harabasz": float("nan"),
            "davies_bouldin": float("nan"),
        }
    return {
        "n_clusters": len(set(labs.tolist())),
        "silhouette": float(silhouette_score(data, labs)),
        "calinski_harabasz": float(calinski_harabasz_score(data, labs)),
        "davies_bouldin": float(davies_bouldin_score(data, labs)),
    }


def agreement(labels_a: np.ndarray, labels_b: np.ndarray) -> dict[str, float]:
    """ARI and NMI between two label vectors, ignoring noise (-1)."""
    valid = (labels_a >= 0) & (labels_b >= 0)
    if valid.sum() < 2:
        return {"ari": float("nan"), "nmi": float("nan")}
    return {
        "ari": float(adjusted_rand_score(labels_a[valid], labels_b[valid])),
        "nmi": float(normalized_mutual_info_score(labels_a[valid], labels_b[valid])),
    }


def print_metric_row(name: str, m: dict[str, Any]) -> None:
    """One-line summary of a partition's metrics."""
    print(
        f"  {name:<14} K={m['n_clusters']:>3}  "
        f"sil={m['silhouette']:>7.4f}  "
        f"CH={m['calinski_harabasz']:>9.0f}  "
        f"DB={m['davies_bouldin']:>7.4f}"
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION OUTPUT PATH
# ════════════════════════════════════════════════════════════════════════


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every clustering technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_1 lesson logs its sweep + final-fit metrics to a single
# SQLite store so students can compare runs across techniques after the
# lesson group ends. Mirrors the M5 ex_1 pattern (autoencoder zoo).

CLUSTERING_DB = "sqlite:///mlfp04_ex1_clustering.db"
EXPERIMENT_NAME = "m4_clustering_zoo"


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the clustering ExperimentTracker (kailash-ml 1.5.1)."""
    tracker = await ExperimentTracker.create(store_url=CLUSTERING_DB)
    return tracker, EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(name, float(value))
        if series_metrics:
            for name, values in series_metrics.items():
                for step, value in enumerate(values, start=1):
                    await run.log_metric(name, float(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 1.2: Hierarchical Clustering with Four Linkage Methods
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build an agglomerative hierarchy bottom-up via the linkage matrix
#   - Compare single, complete, average, and Ward's linkage behaviours
#   - Read a dendrogram and cut it at a chosen K
#   - Choose linkage based on cluster shape expectations
#
# PREREQUISITES: 01_kmeans.py (for the best_k heuristic and feature setup).
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — agglomerative merging and what linkage means
#   2. Build — fit four linkage methods on a subsample
#   3. Train — score each linkage partition against the others
#   4. Visualise — the Ward dendrogram
#   5. Apply — Singapore NTUC FairPrice store-cluster taxonomy
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import numpy as np
from dotenv import load_dotenv
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)


load_dotenv()



### Kailash-ML ExperimentTracker — every clustering run logs here


In [ ]:
tracker, exp_name = setup_engines()



## THEORY — Agglomerative Merging and Linkage


In [ ]:
# Agglomerative clustering starts with every point as its own cluster and
# merges the two closest clusters at each step until one cluster remains.
# The entire merge history is the "dendrogram". To get K clusters, cut
# the dendrogram at a height that leaves K branches.
#
# The open question: "closest" between what? A cluster is a SET of points,
# so cluster-to-cluster distance needs a definition. That definition is
# the LINKAGE method. Four common choices:
#
#   Single   d(A,B) = min over all a∈A, b∈B of ||a-b||   → elongated chains
#   Complete d(A,B) = max over all a∈A, b∈B of ||a-b||   → tight spheres
#   Average  d(A,B) = mean over all pairs                 → balanced
#   Ward's   d(A,B) = increase in within-cluster variance → K-means-like
#
# Ward's is the usual production default. Single linkage is prone to
# "chaining" where two dense clusters get merged because one noisy point
# bridges them. Complete linkage is sensitive to outliers.



## TASK 2 — BUILD: Subsample and fit four linkage methods


Fit one linkage method and score it at the CUT_K cut.


In [ ]:
# Hierarchical is O(n²) memory and O(n² log n) time. Subsample for the
# dendrogram; you can KNN-extend the labels back to full-data if needed.

customers, feature_cols = load_customers()
X_scaled, _ = standardise(customers, feature_cols)
n_samples = X_scaled.shape[0]

# Subsample for dendrogram readability and memory
X_hier, idx_hier = subsample(X_scaled, n=2000, seed=RANDOM_STATE)
n_hier = X_hier.shape[0]

# Cut at K=5 (a reasonable default for 6-dim customer data)
CUT_K = 5
LINKAGE_METHODS = ["single", "complete", "average", "ward"]

print("=" * 70)
print("  Hierarchical Clustering on Singapore E-commerce Customers")
print("=" * 70)
print(f"  Subsample: {n_hier:,} of {n_samples:,} customers")
print(f"  Cut height: K={CUT_K}")
print(
    f"\n  {'Linkage':<10} {'K':>4} {'Silhouette':>12} {'CH':>10} {'DB':>8} {'Time':>8}"
)
print("  " + "─" * 55)


def fit_linkage(method: str) -> dict:
    t0 = time.perf_counter()
    Z = linkage(X_hier, method=method, metric="euclidean")
    elapsed = time.perf_counter() - t0

    labels = fcluster(Z, t=CUT_K, criterion="maxclust") - 1
    k_actual = len(set(labels.tolist()))
    if k_actual >= 2:
        sil = silhouette_score(X_hier, labels)
        ch = calinski_harabasz_score(X_hier, labels)
        db = davies_bouldin_score(X_hier, labels)
    else:
        sil, ch, db = -1.0, 0.0, float("inf")

    return {
        "Z": Z,
        "labels": labels,
        "n_clusters": k_actual,
        "silhouette": sil,
        "ch": ch,
        "db": db,
        "time": elapsed,
    }


hier_results = {m: fit_linkage(m) for m in LINKAGE_METHODS}
for m, r in hier_results.items():
    print(
        f"  {m:<10} {r['n_clusters']:>4} {r['silhouette']:>12.4f} "
        f"{r['ch']:>10.0f} {r['db']:>8.4f} {r['time']:>7.2f}s"
    )



### Checkpoint 1


In [ ]:
assert len(hier_results) == 4, "Task 2: all four linkage methods required"
assert all("silhouette" in r for r in hier_results.values()), "Task 2: scoring gap"
print("\n  [ok] Checkpoint 1 passed — four linkage methods fitted\n")



## TASK 3 — TRAIN: Which linkage wins?


In [ ]:
# Hierarchical does not have a training loop — each linkage method fits in
# closed form given the distance matrix. "Training" here means picking the
# method with the best score AND the best shape match for the data.

best_method = max(hier_results.items(), key=lambda x: x[1]["silhouette"])
print(
    f"  Best linkage by silhouette: {best_method[0]} "
    f"(sil={best_method[1]['silhouette']:.4f})"
)

ward_sil = hier_results["ward"]["silhouette"]
single_sil = hier_results["single"]["silhouette"]
print(f"  Ward's silhouette: {ward_sil:.4f}")
print(
    f"  Single silhouette: {single_sil:.4f}  "
    f"({'chains' if single_sil < ward_sil - 0.05 else 'competitive'})"
)



### Checkpoint 2


In [ ]:
assert best_method[0] in LINKAGE_METHODS, "Task 3: best method selection invalid"
assert (
    hier_results["ward"]["n_clusters"] == CUT_K
), "Task 3: Ward cut should give CUT_K clusters"
print("\n  [ok] Checkpoint 2 passed — Ward's linkage partition extracted\n")



## TASK 4 — VISUALISE: Ward dendrogram (the definitive hierarchical plot)


In [ ]:
# We render the Ward dendrogram directly with scipy's matplotlib helper
# because dendrograms have bespoke geometry that general-purpose plotters
# cannot express. The key features to read:
#   - Y-axis height = merge distance (how dissimilar the merged clusters were)
#   - Large vertical gaps = "natural" cluster boundaries
#   - Horizontal cuts = a specific partition at that height

try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(12, 6))
    dendrogram(
        hier_results["ward"]["Z"],
        truncate_mode="lastp",
        p=30,
        ax=ax,
        leaf_font_size=9,
        color_threshold=0.7 * hier_results["ward"]["Z"][-CUT_K, 2],
    )
    ax.set_title(f"Ward Dendrogram — cut at K={CUT_K}")
    ax.set_xlabel("Cluster size (leaves)")
    ax.set_ylabel("Merge distance")
    fig.tight_layout()
    fig.savefig(str(out_path("02_hier_ward_dendrogram.png")), dpi=120)
    plt.close(fig)
    print(f"  Saved: {out_path('02_hier_ward_dendrogram.png')}")
except ImportError as e:  # pragma: no cover
    raise ImportError(
        "02_hierarchical.py requires matplotlib: "
        "uv add matplotlib  (or  pip install kailash-ml[viz])"
    ) from e

print("\n  Reading the dendrogram:")
print("    * Each horizontal bar is one merge between two clusters")
print("    * Bar HEIGHT = the distance between those two clusters")
print("    * A cut at any height gives you a partition (count the branches)")
print("    * Long vertical gaps above a cut = robust, 'natural' K")



### Checkpoint 3


In [ ]:
assert out_path(
    "02_hier_ward_dendrogram.png"
).exists(), "Task 4: dendrogram not written"
print("\n  [ok] Checkpoint 3 passed — dendrogram rendered\n")



## TASK 5 — APPLY: NTUC FairPrice Store-Cluster Taxonomy


In [ ]:
# SCENARIO: NTUC FairPrice (Singapore's largest supermarket chain) runs
# ~230 stores across Xtra, Finest, Value, and express formats. The
# merchandising team wants a data-driven taxonomy of store behaviour —
# which stores move together on promotions, which are dead-zones for
# non-grocery, which over-index on fresh produce.
#
# Why hierarchical is the right tool here:
#   - 230 stores is a TINY dataset — O(n² log n) is trivial
#   - Merchandising thinks in a TREE ("all Finest stores" contains a
#     sub-branch "Finest with strong wine sales") — a dendrogram is the
#     natural representation of their mental model
#   - Different K values are useful for different decisions (4 clusters
#     for national campaigns, 12 clusters for regional planograms)
#   - Ward's linkage matches their expectation of compact store-type groups
#
# BUSINESS IMPACT: NTUC's public sustainability report discloses ~S$3.2B
# in annual revenue. Category-level promotion mix-ups (stocking the wrong
# SKU ratio per cluster) wastes an estimated 1.5-2% of promotional spend.
# FairPrice spends ~4% of revenue on trade promotion (S$128M/year). Data-
# driven store clustering reduces waste by ~10%:
#     S$128M × 0.10 = S$12.8M / year recovered promotional budget
# The tree-structured taxonomy also lets the team explore "what if we cut
# at K=8 instead of K=5" without refitting — one-time cost, permanent asset.

print("  APPLY — NTUC FairPrice Store Taxonomy")
print("  ─────────────────────────────────────────────────────────────────")
ward_labels = hier_results["ward"]["labels"]
sizes = np.bincount(ward_labels)
for i, n in enumerate(sizes):
    print(f"    Ward cluster {i}: {n:>5,} customers ({n/n_hier:6.1%})")
print("    (In the FairPrice scenario each node is a STORE, not a customer.)")
print("    Estimated annual promo waste recovery: S$12.8M (10% of S$128M).")



### Checkpoint 4


In [ ]:
assert int(sizes.sum()) == n_hier, "Task 5: Ward partition size mismatch"
assert len(sizes) == CUT_K, "Task 5: Ward cut should yield exactly CUT_K clusters"
print("\n  [ok] Checkpoint 4 passed — Ward taxonomy valid\n")



## TRACK — Log this lesson's run to the kailash-ml ExperimentTracker


In [ ]:
track_run(
    tracker,
    exp_name,
    run_name=f"hierarchical_{best_method[0]}",
    params={
        "linkage_methods": ",".join(LINKAGE_METHODS),
        "cut_k": CUT_K,
        "best_linkage": best_method[0],
        "n_subsample": n_hier,
        "n_features": X_hier.shape[1],
    },
    scalar_metrics={
        f"{m}_silhouette": float(r["silhouette"]) for m, r in hier_results.items()
    }
    | {f"{m}_calinski_harabasz": float(r["ch"]) for m, r in hier_results.items()}
    | {f"{m}_davies_bouldin": float(r["db"]) for m, r in hier_results.items()}
    | {f"{m}_time_s": float(r["time"]) for m, r in hier_results.items()},
)
print(
    f"  [tracked] linkage comparison logged to {exp_name} run='hierarchical_{best_method[0]}'\n"
)



## DESTINATION-FIRST CLOSE — engine surface honesty for hierarchical


In [ ]:
# kailash-ml 1.5.1 ClusteringEngine ships kmeans/dbscan/spectral/gmm but
# NOT hierarchical/agglomerative — it's the one mainstream clustering
# family the engine doesn't yet wrap. The engine-first surface for THIS
# lesson is therefore the ExperimentTracker we just used: every linkage
# method, every score, every timing — already in m4_clustering_zoo.db
# next to the kmeans run from 01. The destination is the comparable
# leaderboard, not a one-line replacement for scipy's `linkage`.

from kailash_ml.engines.clustering import ClusteringEngine

print("  ClusteringEngine 1.5.1 algorithms:", ClusteringEngine.__doc__ or "")
print("    Supported: kmeans, dbscan, spectral, gmm")
print(
    "    Hierarchical / agglomerative: use scipy.cluster.hierarchy until"
    " a future kailash-ml release adds the adapter.\n"
)
print(
    "  Engine-first take-away: the tracker leaderboard IS the destination —"
    " open mlfp04_ex1_clustering.db to compare every linkage method against"
    " the kmeans run from lesson 01 side-by-side.\n"
)



## REFLECTION


[x] Agglomerative merging builds a dendrogram bottom-up
  [x] Four linkage methods produce different cluster shapes:
      single (chains), complete (spheres), average (balanced), Ward (variance)
  [x] Read a dendrogram: height = merge distance; cut = partition
  [x] Ward's is the production default for compact, K-means-like clusters
  [x] Mapped the tree onto an NTUC FairPrice store taxonomy with an
      estimated S$12.8M / year promotional-waste recovery

  KEY INSIGHT: When the business thinks in a TREE, give them a tree.
  K-means forces a single K; a dendrogram lets a team explore many K
  values in one fit and pick the granularity that matches the decision.

  Next: 03_density_based.py — clusters of arbitrary SHAPE, not just size.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

